# 05 — Model Baselines
Trains each model individually (linear regression → random forest → SVR → LightGBM) on the same group-aware holdout split, and inspects predicted-vs-actual RUL per model. For the fair, cross-validated comparison across all models at once, see `07_model_comparison.ipynb` — this notebook is for looking closely at *one* model's behavior at a time.

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

import config
import preprocessing
import feature_engineering as fe
import metrics
from models import MODEL_REGISTRY

plt.rcParams['figure.figsize'] = (10, 4)

## Load features and split
Uses the cached features written by `03_feature_engineering.ipynb` (or `feature_engineering.py` directly) — same group-aware split logic as `train.py`, never a random row split.

In [ ]:
if not os.path.exists(config.FEATURES_DATA_PATH):
    preprocessing.run_preprocessing()
    feat_df = fe.run_feature_engineering()
else:
    feat_df = pd.read_csv(config.FEATURES_DATA_PATH, parse_dates=[config.COL_TIMESTAMP])

train_df, test_df = preprocessing.split_data(feat_df)
feature_cols = fe.get_feature_columns(feat_df)

X_train, y_train = train_df[feature_cols], train_df[config.COL_RUL]
X_test, y_test = test_df[feature_cols], test_df[config.COL_RUL]

## Train each baseline and collect predictions
Same wrapper classes `train.py` uses — this loop is the notebook equivalent of running `python train.py --model X` for each model in sequence, but keeps everything in memory for plotting.

In [ ]:
predictions = {}
all_scores = {}

for name, model_cls in MODEL_REGISTRY.items():
    wrapper = model_cls()
    Xtr, Xte = X_train.copy(), X_test.copy()

    if wrapper.needs_scaling:
        scaler = StandardScaler()
        Xtr = pd.DataFrame(scaler.fit_transform(Xtr), columns=feature_cols, index=Xtr.index)
        Xte = pd.DataFrame(scaler.transform(Xte), columns=feature_cols, index=Xte.index)

    wrapper.fit(Xtr, y_train)
    y_pred = wrapper.predict(Xte)

    predictions[name] = y_pred
    scores = metrics.evaluate(y_test, y_pred)
    all_scores[name] = scores
    metrics.print_scores(name, scores)

In [ ]:
scores_df = pd.DataFrame(all_scores).T[['MAE', 'RMSE', 'R2', 'PHM08']]
scores_df.sort_values('PHM08')

## Predicted vs. actual RUL, per model
The diagonal line is perfect prediction. Points above the line are late/optimistic predictions (the operationally dangerous direction per the PHM08 score); points below are early/conservative.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 10))
for ax, (name, y_pred) in zip(axes.ravel(), predictions.items()):
    ax.scatter(y_test, y_pred, s=6, alpha=0.3)
    lims = [0, max(y_test.max(), max(y_pred))]
    ax.plot(lims, lims, 'r--', linewidth=1)
    ax.set_title(name)
    ax.set_xlabel('Actual RUL')
    ax.set_ylabel('Predicted RUL')
plt.tight_layout()
plt.show()

## Residuals over predicted RUL
Check whether error grows as RUL approaches zero — that's the region where prediction accuracy matters most operationally, and models can look good on aggregate RMSE while being worst exactly there.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 10))
for ax, (name, y_pred) in zip(axes.ravel(), predictions.items()):
    residual = np.array(y_pred) - y_test.values
    ax.scatter(y_test, residual, s=6, alpha=0.3)
    ax.axhline(0, color='red', linewidth=1, linestyle='--')
    ax.set_title(name)
    ax.set_xlabel('Actual RUL')
    ax.set_ylabel('Prediction error (pred - actual)')
plt.tight_layout()
plt.show()

## Next step
This notebook uses a single holdout split, so ranking here can be noisy with few units. Proceed to `07_model_comparison.ipynb` for the GroupKFold cross-validated comparison before picking a model.